# Agent vs Chatbot vs Workflow

Same customer query, three implementations. We'll feel why agents win when the path isn't predictable.

**Query under test:** `"Customer wants a refund on damaged order #ORDNO5 — what can we do?"`

In [3]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

llm = ChatOllama(model="qwen3.5:2b", temperature=0.7)

QUERY_A = "Customer wants a refund on damaged order #ORDNO5 — what can we do?"
QUERY_B = "Where is my package? Order #ORDNO7."

## 1. Chatbot — `prompt → LLM → answer`

A plain LLM call. No tools, no context. It can only talk, not act.

In [4]:
def chatbot(query: str) -> str:
    return llm.invoke(query).content

print("--- Query A (refund) ---")
print(chatbot(QUERY_A))
print("\n--- Query B (where is package) ---")
print(chatbot(QUERY_B))

--- Query A (refund) ---
Here is a step-by-step guide on how to handle this request professionally and efficiently:

### 1. Immediate Verification & Acknowledgment
*   **Verify Order ID:** Confirm that `ORDNO5` exists in your system (e.g., check the database or backend). Ensure it matches the shipping date and customer details.
*   **Acknowledge the Claim:** Send a message like, *"Thank you for contacting us regarding order #ORDNO5. We understand this is an issue with the packaging/delivery."* This shows empathy and sets a professional tone.

### 2. Assess the Damage & Evidence
*   **Inspect the Package:** Check if the damage was caused by shipping (e.g., dropped, crushed) or handling issues (e.g., broken item inside).
*   **Gather Documentation:** Ask the customer to take photos/videos of the damaged item and any packaging marks. This is crucial for policy validation and warranty claims.
*   **Check Warranty Status:** Determine if this order was covered by manufacturer warranty, insur

**Verdict:** Generic empathy. No real lookup. No real policy. No action taken.

## 2. Workflow — `prompt + context → LLM → answer`

Hardcoded pipeline. A developer decides the steps in advance. Works when the path is fixed — breaks the moment the query deviates.

We reuse the `lookupOrder` and `check_policy` tools from Notebook 1, but here we call them ourselves in a fixed order.

In [5]:
def lookup_order(order_no: str) -> str:
    df = pd.read_csv("order_mock_data.csv")
    if not order_no.startswith("#"):
        order_no = "#" + order_no
    row = df[df["id"] == order_no]
    if row.empty:
        return f"No order found with ID {order_no}."
    r = row.iloc[0]
    return f"OrderNo: {r['id']}, Name: {r['name']}, Status: {r['status']}, Items: {r['items']}"

def build_retriever():
    loader = PyPDFLoader("sample_policy.pdf")
    pages = loader.load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectordb = Chroma.from_documents(chunks, embedding=embeddings, persist_directory="./chroma.db")
    return vectordb.as_retriever(search_kwargs={"k": 3})

retriever = build_retriever()

def check_policy(query: str) -> str:
    chunks = retriever.invoke(query)
    return "\n\n".join([c.page_content for c in chunks])

In [6]:
import re

def workflow(query: str) -> str:
    """Hardcoded refund pipeline. The dev wrote the steps; the LLM just narrates."""
    if "refund" not in query.lower():
        return "Workflow only handles refund requests. Please rephrase."

    # Step 1: extract order ID
    m = re.search(r"#ORDNO\d+", query)
    if not m:
        return "No order ID found in query."
    order_id = m.group()

    # Step 2: lookup
    order_info = lookup_order(order_id)

    # Step 3: policy
    policy = check_policy("damaged product refund")

    # Step 4: LLM composes a reply from the fetched context
    prompt = (
        f"Customer query: {query}\n\n"
        f"Order info: {order_info}\n\n"
        f"Relevant policy:\n{policy}\n\n"
        "Write a short, helpful reply to the customer."
    )
    return llm.invoke(prompt).content

In [7]:
print("--- Query A (refund) — workflow handles this ---")
print(workflow(QUERY_A))
print("\n--- Query B (where is package) — workflow breaks ---")
print(workflow(QUERY_B))

--- Query A (refund) — workflow handles this ---
Hi Charles, thank you for contacting us regarding Order #ORDNO5. Since your item is currently in "Shipping" status and you've reported damage, we can process a refund under our damaged in transit policy. We will require a photo of the damage for verification, and restocking fees are waived. Return shipping costs will be covered by us, and refunds will be processed within SLAS guidelines (typically 24 hours). Please let us know once you have the photo, and we'll get back to you shortly.

--- Query B (where is package) — workflow breaks ---
Workflow only handles refund requests. Please rephrase.


**Verdict:** Query A works because the path (extract → lookup → policy → compose) was pre-built for exactly that shape. Query B hits the `if "refund" not in query` branch and exits. A developer would have to add a new branch for every new intent.

## 3. Agent — `prompt → LLM ⇄ {tools, memory} → action`

The LLM decides which tools to call, in what order, based on the query. Same two tools, but now the *path* is chosen at runtime.

In [8]:
@tool
def lookupOrder(orderNo: str) -> str:
    """Look up order status by order ID. Order IDs look like #ORDNO1, #ORDNO2, etc."""
    return lookup_order(orderNo)

@tool
def checkPolicy(query: str) -> str:
    """Query the company policy on order issues, refunds, damages, shipping, returns."""
    return check_policy(query)

tools = [lookupOrder, checkPolicy]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

In [9]:
def run_agent(query: str, max_iters: int = 6) -> str:
    messages = [HumanMessage(content=query)]
    for _ in range(max_iters):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for call in response.tool_calls:
            result = tool_map[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "Max iterations reached."

In [10]:
print("--- Query A (refund) ---")
print(run_agent(QUERY_A))
print("\n--- Query B (where is package) ---")
print(run_agent(QUERY_B))

--- Query A (refund) ---
Based on the order status and company policy, here's what you can do:

## Current Order Status
**Order #ORDNO5 (Charles Martinez)** is currently in **Shipping** status. It contains:
- Fitness Tracker
- Desk Lamp

## Your Options

### 1. **Check for Delivery Delay**
Since the order is still in transit, first verify if there's a delay:
- Check carrier tracking system to confirm if the order has been delayed
- Review the estimated delivery timeframe promised at checkout
- If the package hasn't arrived by the expected date, this qualifies as a "late order delivery"

### 2. **Refund Process (if damaged)**
If the damage is confirmed:
- The refund will be processed based on the warehouse's inspection grade
- Refunds are typically issued within **48 hours** of receipt
- Finance processes the refund after the warehouse completes its inspection

### 3. **Immediate Actions**
- **Contact the customer** to confirm if there was damage during shipping
- **Provide tracking inf

**Verdict:** The agent handled both queries. For A, it called `lookupOrder` *and* `checkPolicy`. For B, it called just `lookupOrder`. Nobody hardcoded that decision — the LLM made it at runtime from the tool descriptions.

## Summary

| | Chatbot | Workflow | Agent |
|---|---|---|---|
| Pattern | `prompt → LLM → answer` | `prompt + context → LLM → answer` | `prompt → LLM ⇄ {tools} → action` |
| Who decides the path? | Nobody — no path | Developer, in advance | LLM, at runtime |
| Handles Query A | ❌ no action | ✅ | ✅ |
| Handles Query B | ❌ no action | ❌ wrong branch | ✅ |
| Limit | Can't act on the world | Can't deviate from script | Costs more tokens, harder to debug |

**Rule of thumb:** if the path is predictable, use a workflow. If the path depends on the query, use an agent.